<font color='rainbow' size=6pt> Supervised Machine Learning: Match Predictor

In [ ]:
#set up environment
!python set_up.py

import os
pip_upgrade = os.path.join("venv", "Scripts", "python.exe") if os.name == "nt" else os.path.join("venv", "bin", "python")
!{pip_upgrade} -m pip install --upgrade pip

!pip install -r https://raw.githubusercontent.com/Keegan-McCullough/Soccer_Match_Predictor/main/requirements.txt

In [ ]:
import kagglehub
import pandas as pd
import os
import sqlite3

# Download epl and europe datasets from Kaggle
epl_path = kagglehub.dataset_download("saife245/english-premier-league")

# Load EPL CSVs into a dictionary of DataFrames
epl_dfs = {}
for f in os.listdir(epl_path):
    if f.endswith(".csv"):
        epl_dfs[f] = pd.read_csv(os.path.join(epl_path, f))

# Download Europe soccer SQLite database
europe_path = kagglehub.dataset_download("hugomathien/soccer")

# Find the file
db_file = [f for f in os.listdir(europe_path) if f.endswith(".sqlite")][0]
db_path = os.path.join(europe_path, db_file)

# Connect and list all tables
conn = sqlite3.connect(db_path)
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
# Load each table into a dictionary of DataFrames
europe_dfs = {}
for table in tables["name"]:
    europe_dfs[table] = pd.read_sql(f"SELECT * FROM {table}", conn)
conn.close()

In [ ]:
# EPL data structure overview
print("EPL datasets:", list(epl_dfs.keys()))
print(epl_dfs["final_dataset.csv"].head())

In [ ]:
# Europe DB tables overview
print("\nEurope DB tables:", tables["name"].tolist())
english_teams = ['Arsenal','Aston Villa', 'Blackburn Rovers', 'Bolton Wanderers', 
                 'Chelsea', 'Everton', 'Liverpool', 'Manchester City', 'Manchester United',
                 'Middlesbrough', 'Newcastle United', 'Queens Park Rangers' 'Southampton',
                 'Tottenham Hotspur', 'West Ham United', 'Burnley', 'Swansea', 'Wolverhampton Wanderers'
                 'Cardiff City', 'Hull City', 'Stoke City', 'Norwich City', 'West Bromwich Albion',
                 'Blackpool', 'Crystal Palace', 'Watford']

df_teams = europe_dfs["Team"]
team_map = {}
for index, team in df_teams.iterrows():
    if team['team_long_name'] in english_teams:
        team_map[team['team_long_name']] = team['team_api_id']

